# 🔍 Financial Transaction Fraud Detection
### Accredian Internship Task — Data Science & Machine Learning

---

**Dataset:** PaySim Synthetic Financial Transactions  
**Rows:** 6,362,620 &nbsp;|&nbsp; **Columns:** 10  
**Target:** `isFraud` (1 = fraudulent, 0 = legitimate)

---

## Section 1 — Problem Understanding

Financial fraud is one of the most costly problems facing modern banks and payment platforms. According to industry reports, global payment fraud losses run into the tens of billions annually. The challenge is not just finding fraud — it's finding it **fast enough** and **accurately enough** to act.

Two types of errors carry very different costs in this domain:

| Error Type | What happens | Business Cost |
|---|---|---|
| **False Negative** (missed fraud) | A fraudulent transaction is approved | Direct financial loss, regulatory risk |
| **False Positive** (good transaction blocked) | A legitimate transaction is declined | Customer frustration, churn, support costs |

Neither is acceptable in large numbers. A model with 99.9% accuracy sounds great — until you realise that predicting *everything* as legitimate achieves 99.87% accuracy on this dataset. The real question is: **can the model catch most fraud without killing the customer experience?**

This notebook approaches the problem as a data scientist would in practice: start with curiosity about the data, investigate patterns, engineer useful signals, and build models that optimise for the right objectives — not just headline accuracy.

## Section 2 — Environment Setup

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
})
FRAUD_COLOR  = '#e74c3c'
LEGIT_COLOR  = '#2980b9'
ACCENT_COLOR = '#f39c12'

# ── Preprocessing ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline

# ── Models ────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)

# ── Imbalance handling ────────────────────────────────────────────────────────
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    IMBLEARN = True
    print("✅ imbalanced-learn available")
except ImportError:
    IMBLEARN = False
    print("⚠️  imbalanced-learn not installed — using class_weight='balanced'")
    print("    Install with: pip install imbalanced-learn")

# ── Multicollinearity ─────────────────────────────────────────────────────────
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    VIF_AVAILABLE = True
except ImportError:
    VIF_AVAILABLE = False

import os, time
print(f"\nPandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")
print("\n✅ Setup complete")

## Section 3 — Data Loading & Initial Exploration

The dataset is the **PaySim** synthetic mobile-money simulation, originally published on Kaggle.  
Download link: https://www.kaggle.com/datasets/ealaxi/paysim1

**Column descriptions:**
| Column | Description |
|---|---|
| `step` | Hour of simulation (1 step = 1 hour, 744 steps total = 30 days) |
| `type` | Transaction type: CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER |
| `amount` | Transaction amount (local currency) |
| `nameOrig` | Customer initiating the transaction |
| `oldbalanceOrg` | Sender balance **before** transaction |
| `newbalanceOrig` | Sender balance **after** transaction |
| `nameDest` | Recipient of the transaction |
| `oldbalanceDest` | Recipient balance **before** transaction |
| `newbalanceDest` | Recipient balance **after** transaction |
| `isFraud` | Ground truth label (1 = fraud) |
| `isFlaggedFraud` | Legacy rule-based flag (transfer > 200,000) |

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
# Update FILE_PATH to point to your downloaded CSV
FILE_PATH = 'PS_20174392719_1491204439457_log.csv'

t0 = time.time()
df = pd.read_csv(FILE_PATH)
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory  : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
df.head()

In [ ]:
print("Dtypes:")
print(df.dtypes)
print("\nBasic statistics:")
df.describe().T.round(2)

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
fraud_counts = df['isFraud'].value_counts()
fraud_pct    = df['isFraud'].value_counts(normalize=True) * 100

print("Target variable distribution:")
summary = pd.DataFrame({'Count': fraud_counts, 'Pct (%)': fraud_pct.round(4)})
summary.index = ['Legitimate (0)', 'Fraud (1)']
print(summary)

print(f"\nImbalance ratio: 1 fraud per {fraud_counts[0]//fraud_counts[1]:,} legitimate transactions")

In [ ]:
# ── Transaction types ─────────────────────────────────────────────────────────
type_fraud = df.groupby('type')['isFraud'].agg(['sum','count'])
type_fraud.columns = ['Fraud_Count', 'Total']
type_fraud['Fraud_Rate_%'] = (type_fraud['Fraud_Count'] / type_fraud['Total'] * 100).round(3)
print("Fraud by transaction type:")
print(type_fraud.sort_values('Fraud_Rate_%', ascending=False))

**Observation:** Fraud only occurs in `TRANSFER` and `CASH_OUT` transactions. This is a very important signal — we can immediately restrict model training and deployment to these two types, which will sharply reduce false positives from irrelevant transaction categories.

## Section 4 — Data Cleaning & Preparation

### 4.1 Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100
print("Missing values:")
result = pd.DataFrame({'Count': missing, 'Pct%': missing_pct.round(4)})
print(result[result['Count'] > 0] if missing.sum() > 0 else "✅ No missing values found.")

### 4.2 Remove Redundant / High-Cardinality Columns

In [ ]:
# nameOrig and nameDest are unique customer IDs — millions of distinct values,
# no generalizable signal. isFlaggedFraud is a hard-coded rule (TRANSFER > 200k)
# that misses the vast majority of actual fraud. Both are dropped.

print(f"Unique nameOrig : {df['nameOrig'].nunique():,}")
print(f"Unique nameDest : {df['nameDest'].nunique():,}")
print(f"isFlaggedFraud  : {df['isFlaggedFraud'].sum():,} flags vs {df['isFraud'].sum():,} actual fraud cases")

df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'], inplace=True)
print(f"\nColumns after drop: {df.columns.tolist()}")

### 4.3 Outlier Detection & Treatment

In [ ]:
# IQR-based outlier summary for numeric features
# We will NOT blindly remove outliers here — fraud itself often manifests as
# unusually large transactions. Instead we inspect and cap at the 99th percentile
# to tame extreme skew without deleting potentially informative data.

num_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

print("Outlier summary (IQR method):")
print(f"{'Column':<20} {'Q1':>12} {'Q3':>12} {'IQR':>12} {'Upper Fence':>14} {'Max':>14} {'Outliers':>10}")
print("-" * 90)
for col in num_cols:
    q1  = df[col].quantile(0.25)
    q3  = df[col].quantile(0.75)
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr
    n_out = (df[col] > upper_fence).sum()
    print(f"{col:<20} {q1:>12.0f} {q3:>12.0f} {iqr:>12.0f} {upper_fence:>14.0f} {df[col].max():>14.0f} {n_out:>10,}")

In [ ]:
# Cap at 99th percentile — preserves data volume while reducing extreme skew
# Note: We apply this ONLY AFTER feature engineering so we don't lose signal
# in the engineered balance-difference features.

cap_info = {}
for col in num_cols:
    p99 = df[col].quantile(0.99)
    p01 = df[col].quantile(0.01)
    cap_info[col] = (p01, p99)
    df[col] = df[col].clip(lower=p01, upper=p99)
    print(f"Capped {col:<22} → [{p01:>12.0f}, {p99:>12.0f}]")

print("\n✅ Outlier capping complete.")

### 4.4 Multicollinearity Check (VIF)

High multicollinearity inflates coefficient variance in linear models and complicates interpretation. We compute Variance Inflation Factor (VIF) — values above 10 are concerning.

In [ ]:
if VIF_AVAILABLE:
    tmp = df.copy()
    tmp['type_enc'] = LabelEncoder().fit_transform(tmp['type'])
    vif_cols = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
                'oldbalanceDest', 'newbalanceDest', 'type_enc']
    vif_matrix = tmp[vif_cols].dropna()

    vif_df = pd.DataFrame({
        'Feature': vif_cols,
        'VIF': [variance_inflation_factor(vif_matrix.values, i) for i in range(len(vif_cols))]
    }).sort_values('VIF', ascending=False)

    print("Variance Inflation Factor:")
    print(vif_df.to_string(index=False))
    print("\n→ oldbalanceOrg/newbalanceOrig and oldbalanceDest/newbalanceDest")
    print("  show high VIF. We'll replace them with difference features.")
else:
    print("Install statsmodels for VIF: pip install statsmodels")
    print("Intuition: oldbalanceOrg ≈ newbalanceOrig + amount → strong linear dependence.")

## Section 5 — Exploratory Data Analysis

In [ ]:
# ── 5.1  Class imbalance visualisation ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['isFraud'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values,
            color=[LEGIT_COLOR, FRAUD_COLOR], edgecolor='white', width=0.5)
axes[0].set_title('Transaction Count by Class', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5000, f'{v:,}', ha='center', fontsize=10)

# Pie chart
axes[1].pie(counts.values, labels=['Legitimate', 'Fraud'],
            colors=[LEGIT_COLOR, FRAUD_COLOR],
            autopct='%1.3f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Class Distribution', fontweight='bold')

plt.suptitle('Severe Class Imbalance — Fraud is < 0.2% of Transactions',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("\n→ The extreme imbalance means accuracy is a useless metric here.")
print("  A model predicting all-legitimate would score ~99.87% accuracy.")

In [ ]:
# ── 5.2  Fraud rate by transaction type ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_counts = df.groupby(['type', 'isFraud']).size().unstack(fill_value=0)
type_counts.plot(kind='bar', ax=axes[0], color=[LEGIT_COLOR, FRAUD_COLOR],
                 edgecolor='white', width=0.7)
axes[0].set_title('Transaction Volume by Type & Class', fontweight='bold')
axes[0].set_xlabel('Transaction Type')
axes[0].set_ylabel('Count')
axes[0].legend(['Legitimate', 'Fraud'])
axes[0].tick_params(axis='x', rotation=30)

fraud_rate = (type_counts[1] / type_counts.sum(axis=1) * 100).sort_values(ascending=False)
bars = axes[1].bar(fraud_rate.index, fraud_rate.values, color=ACCENT_COLOR, edgecolor='white')
axes[1].set_title('Fraud Rate (%) by Transaction Type', fontweight='bold')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, fraud_rate.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()
print("\n→ TRANSFER and CASH_OUT are the ONLY fraud-bearing transaction types.")
print("  TRANSFER has a ~0.77% fraud rate — 4x higher than CASH_OUT.")
print("  PAYMENT, CASH_IN, DEBIT: zero fraud. This is a huge modelling constraint.")

In [ ]:
# ── 5.3  Amount distribution: fraud vs legitimate ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fraud_amounts = df[df['isFraud']==1]['amount']
legit_amounts = df[df['isFraud']==0]['amount']

# Log-scale KDE
sns.kdeplot(np.log1p(legit_amounts), ax=axes[0], color=LEGIT_COLOR,
            fill=True, alpha=0.4, label='Legitimate')
sns.kdeplot(np.log1p(fraud_amounts), ax=axes[0], color=FRAUD_COLOR,
            fill=True, alpha=0.4, label='Fraud')
axes[0].set_title('Amount Distribution (log scale)', fontweight='bold')
axes[0].set_xlabel('log(1 + Amount)')
axes[0].legend()

# Box plot
plot_df = pd.DataFrame({
    'log_amount': np.log1p(df['amount']),
    'Class': df['isFraud'].map({0: 'Legitimate', 1: 'Fraud'})
})
sns.boxplot(x='Class', y='log_amount', data=plot_df,
            palette={'Legitimate': LEGIT_COLOR, 'Fraud': FRAUD_COLOR},
            ax=axes[1], width=0.4)
axes[1].set_title('Amount Box Plot by Class', fontweight='bold')
axes[1].set_ylabel('log(1 + Amount)')

plt.suptitle('Fraud Transactions Tend to Involve Higher Amounts', fontsize=13,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Median amount — Legitimate: {legit_amounts.median():,.0f}")
print(f"Median amount — Fraud     : {fraud_amounts.median():,.0f}")
print("\n→ Fraudulent transactions are significantly larger on average.")

In [ ]:
# ── 5.4  Temporal pattern of fraud ──────────────────────────────────────────
hourly = df.groupby('step')['isFraud'].agg(['mean', 'sum']).reset_index()
hourly.columns = ['step', 'fraud_rate', 'fraud_count']

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].fill_between(hourly['step'], hourly['fraud_count'],
                     color=FRAUD_COLOR, alpha=0.7)
axes[0].set_ylabel('Fraud Count')
axes[0].set_title('Fraud Volume Over Time (by simulation hour)', fontweight='bold')

axes[1].fill_between(hourly['step'], hourly['fraud_rate'] * 100,
                     color=ACCENT_COLOR, alpha=0.7)
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_xlabel('Simulation Hour (Step)')
axes[1].set_title('Fraud Rate Over Time', fontweight='bold')

plt.tight_layout()
plt.show()
print("\n→ Fraud shows temporal clustering — elevated activity in certain hour windows.")
print("  This could correspond to off-hours or periods of reduced manual oversight.")

In [ ]:
# ── 5.5  Balance anomalies ────────────────────────────────────────────────────
# A suspicious pattern: sender balance doesn't drop despite a large transfer.
# This can indicate data manipulation or testing how fraud was constructed.

sample = df.sample(min(50000, len(df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in [(0, LEGIT_COLOR), (1, FRAUD_COLOR)]:
    sub = sample[sample['isFraud'] == label]
    axes[0].scatter(np.log1p(sub['oldbalanceOrg']),
                    np.log1p(sub['newbalanceOrig']),
                    alpha=0.3, s=5, c=color,
                    label='Legitimate' if label == 0 else 'Fraud')

axes[0].set_xlabel('log(1 + Old Balance Orig)')
axes[0].set_ylabel('log(1 + New Balance Orig)')
axes[0].set_title('Sender: Old vs New Balance', fontweight='bold')
axes[0].legend(markerscale=3)
axes[0].plot([0,15],[0,15], 'k--', lw=1, alpha=0.4, label='Equal line')

for label, color in [(0, LEGIT_COLOR), (1, FRAUD_COLOR)]:
    sub = sample[sample['isFraud'] == label]
    axes[1].scatter(np.log1p(sub['oldbalanceDest']),
                    np.log1p(sub['newbalanceDest']),
                    alpha=0.3, s=5, c=color,
                    label='Legitimate' if label == 0 else 'Fraud')

axes[1].set_xlabel('log(1 + Old Balance Dest)')
axes[1].set_ylabel('log(1 + New Balance Dest)')
axes[1].set_title('Recipient: Old vs New Balance', fontweight='bold')
axes[1].legend(markerscale=3)
axes[1].plot([0,15],[0,15], 'k--', lw=1, alpha=0.4)

plt.suptitle('Balance Patterns — Fraud Clusters Near Axes (Zero Balance Anomaly)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("\n→ Many fraud cases show recipient's balance NOT increasing after the transaction.")
print("  This 'zero-balance recipient' anomaly is a very strong fraud signal.")

In [ ]:
# ── 5.6  Correlation heatmap ──────────────────────────────────────────────────
tmp_enc = df.copy()
tmp_enc['type_enc'] = LabelEncoder().fit_transform(tmp_enc['type'])
corr_df = tmp_enc.drop(columns=['type']).corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.5,
            annot_kws={'size': 9})
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("\n→ oldbalanceOrg ↔ newbalanceOrig: strong positive correlation (expected).")
print("  These will be replaced by a single balance-change feature.")

## Section 6 — Feature Engineering

Raw balance columns are highly correlated with each other. We'll construct features that capture the **change** and **anomaly** in balances rather than the raw values.

In [ ]:
# ── 6.1  Balance change features ─────────────────────────────────────────────
# If amount was transferred, the sender's balance should drop by exactly 'amount'.
# Discrepancies flag potential data anomalies or fraud masking.

df['balance_diff_orig'] = df['newbalanceOrig'] - df['oldbalanceOrg']
df['balance_diff_dest'] = df['newbalanceDest'] - df['oldbalanceDest']

# Error terms: expected change vs actual change
# For a legitimate TRANSFER: balance_diff_orig ≈ -amount
df['error_orig'] = df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
df['error_dest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

# ── 6.2  Log transformation ───────────────────────────────────────────────────
# Amount is heavily right-skewed. Log1p stabilises variance.
df['log_amount'] = np.log1p(df['amount'])

# ── 6.3  Zero-balance flags ───────────────────────────────────────────────────
# A sender going to exactly zero balance after transaction is suspicious.
df['orig_zero_after']  = (df['newbalanceOrig'] == 0).astype(int)
df['dest_zero_before'] = (df['oldbalanceDest'] == 0).astype(int)

# ── 6.4  Transaction type encoding ───────────────────────────────────────────
# One-hot encode for linear models; we'll also keep a label-encoded version
type_dummies = pd.get_dummies(df['type'], prefix='type', drop_first=True)
df = pd.concat([df, type_dummies], axis=1)
df['type_enc'] = LabelEncoder().fit_transform(df['type'])

print("New features created:")
new_feats = ['balance_diff_orig', 'balance_diff_dest', 'error_orig', 'error_dest',
             'log_amount', 'orig_zero_after', 'dest_zero_before']
print(df[new_feats + ['isFraud']].head())

In [ ]:
# ── Validate that new features have predictive power ─────────────────────────
print("Mean error_orig by fraud class:")
print(df.groupby('isFraud')['error_orig'].mean())
print("\nMean error_dest by fraud class:")
print(df.groupby('isFraud')['error_dest'].mean())
print("\nZero-balance after (orig) fraud rate:")
print(df.groupby('orig_zero_after')['isFraud'].mean().round(4))
print("\n→ error_orig and orig_zero_after are strong discriminators.")

## Section 7 — Feature Selection & Train/Test Split

In [ ]:
# ── Final feature set ─────────────────────────────────────────────────────────
# Rationale:
#  - step: temporal context
#  - log_amount: transaction size (log-transformed for normality)
#  - error_orig/dest: balance accounting discrepancies
#  - orig_zero_after, dest_zero_before: zero-balance anomaly flags
#  - type_* one-hot columns: transaction category
#
# Removed: raw balance columns (high VIF, replaced by error terms)

type_ohe_cols = [c for c in df.columns if c.startswith('type_') and c != 'type_enc']

FEATURES = (['step', 'log_amount',
             'balance_diff_orig', 'balance_diff_dest',
             'error_orig', 'error_dest',
             'orig_zero_after', 'dest_zero_before']
            + type_ohe_cols)

TARGET = 'isFraud'

X = df[FEATURES]
y = df[TARGET]

print(f"Feature matrix shape : {X.shape}")
print(f"Features used        : {FEATURES}")

# ── Stratified split (80/20) ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain : {X_train.shape[0]:,}  |  Fraud in train : {y_train.sum():,} ({y_train.mean()*100:.3f}%)")
print(f"Test  : {X_test.shape[0]:,}   |  Fraud in test  : {y_test.sum():,} ({y_test.mean()*100:.3f}%)")

In [ ]:
# ── Scaling for logistic regression ──────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("✅ Scaling complete (fit on train, applied to test)")

## Section 8 — Handling Class Imbalance

With ~0.13% fraud rate, simply training on the raw distribution almost always results in a model that ignores the minority class. Three strategies are available:

| Strategy | Pros | Cons |
|---|---|---|
| **class_weight='balanced'** | Simple, no data augmentation | May over-penalise majority class |
| **SMOTE oversampling** | Synthetically balances minority class | Can overfit synthetic samples |
| **Undersampling** | Fast training on smaller dataset | Discards potentially useful data |
| **Threshold tuning** | Precise control of FP/FN trade-off | Requires careful post-hoc calibration |

**Our approach:** Use `class_weight='balanced'` in tree-based models (fast and effective), and apply SMOTE on a downsampled training subset for the logistic regression baseline. We'll also demonstrate threshold tuning on the best model.

In [ ]:
# ── Prepare a SMOTE-balanced training set (subset for memory efficiency) ──────
if IMBLEARN:
    # Work on a 20% subsample for SMOTE to avoid memory issues with 6M rows
    X_tr_sub, _, y_tr_sub, _ = train_test_split(
        X_train, y_train, train_size=0.2, random_state=42, stratify=y_train
    )
    print(f"Subsample for SMOTE: {X_tr_sub.shape[0]:,} rows")
    print(f"Fraud before SMOTE : {y_tr_sub.sum():,} ({y_tr_sub.mean()*100:.3f}%)")

    smote = SMOTE(random_state=42, k_neighbors=5)
    X_sm, y_sm = smote.fit_resample(X_tr_sub, y_tr_sub)

    print(f"Shape after SMOTE  : {X_sm.shape}")
    print(f"Fraud after SMOTE  : {y_sm.sum():,} ({y_sm.mean()*100:.1f}%)")
    print("✅ SMOTE complete")
else:
    print("imbalanced-learn not available — using class_weight='balanced' for all models.")
    X_sm, y_sm = X_train, y_train

## Section 9 — Model Building

We build three models with different philosophies:

1. **Logistic Regression** — interpretable linear baseline
2. **Random Forest** — ensemble of decision trees, robust to noise
3. **Gradient Boosting** — sequential error-correction, often best on tabular data

All models are evaluated on the same held-out test set.

In [ ]:
# ── Helper: evaluate a trained classifier ─────────────────────────────────────
def evaluate_model(model, X_test, y_test, model_name, threshold=0.5):
    """Return dict of evaluation metrics and print a formatted summary."""
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    results = {
        'Model':     model_name,
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_prob),
        'Avg-PR':    average_precision_score(y_test, y_prob),
        'y_prob':    y_prob,
        'y_pred':    y_pred,
    }

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(classification_report(y_test, y_pred,
                                target_names=['Legitimate', 'Fraud']))
    print(f"  ROC-AUC  : {results['ROC-AUC']:.4f}")
    print(f"  Avg PR   : {results['Avg-PR']:.4f}")
    return results

In [ ]:
# ── 9.1  Logistic Regression (SMOTE-balanced) ─────────────────────────────────
print("Training Logistic Regression...")
X_sm_scaled = scaler.fit_transform(X_sm)   # refit scaler on SMOTE data

lr_model = LogisticRegression(
    max_iter=500,
    solver='lbfgs',
    class_weight='balanced',
    random_state=42
)
lr_model.fit(X_sm_scaled, y_sm)

# Use the original test scaling for evaluation
X_test_scaled_lr = scaler.transform(X_test)
lr_results = evaluate_model(lr_model, X_test_scaled_lr, y_test, 'Logistic Regression')

In [ ]:
# ── 9.2  Random Forest ────────────────────────────────────────────────────────
# class_weight='balanced_subsample': each tree's bootstrap sample is balanced.
# n_estimators=200 for stability; max_depth=20 prevents overfitting.

print("Training Random Forest... (may take a few minutes)")
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=10,
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)   # tree models work fine with unscaled data
rf_results = evaluate_model(rf_model, X_test, y_test, 'Random Forest')

In [ ]:
# ── 9.3  Gradient Boosting ────────────────────────────────────────────────────
# GBM with subsample and max_features for regularisation.
# We use n_estimators=150 to balance performance vs training time.

print("Training Gradient Boosting... (may take several minutes)")
gb_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    max_features='sqrt',
    random_state=42
)
gb_model.fit(X_train, y_train)
gb_results = evaluate_model(gb_model, X_test, y_test, 'Gradient Boosting')

## Section 10 — Model Evaluation

**Why each metric matters in fraud detection:**

| Metric | Why it matters |
|---|---|
| **Recall** | Catches fraud — a missed fraud is a real financial loss |
| **Precision** | Controls false positives — blocking good customers damages trust |
| **F1-score** | Balances both — useful for model comparison |
| **ROC-AUC** | Rank-ordering ability across all thresholds |
| **PR-AUC** | More informative than ROC-AUC under severe imbalance |

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────────────
all_results = [lr_results, rf_results, gb_results]
compare_df = pd.DataFrame([{
    'Model':     r['Model'],
    'Precision': round(r['Precision'], 4),
    'Recall':    round(r['Recall'], 4),
    'F1':        round(r['F1'], 4),
    'ROC-AUC':   round(r['ROC-AUC'], 4),
    'PR-AUC':    round(r['Avg-PR'], 4),
} for r in all_results])

print("Model Comparison:")
print(compare_df.to_string(index=False))

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#8e44ad', '#27ae60', '#e67e22']
for res, color in zip(all_results, colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f"{res['Model']} (AUC={res['ROC-AUC']:.4f})")

axes[0].plot([0,1],[0,1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=9)

# Precision-Recall curves
for res, color in zip(all_results, colors):
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    axes[1].plot(rec, prec, color=color, lw=2,
                 label=f"{res['Model']} (AP={res['Avg-PR']:.4f})")

baseline = y_test.mean()
axes[1].axhline(baseline, color='gray', linestyle='--', lw=1,
                label=f'Random baseline ({baseline:.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontweight='bold')
axes[1].legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()
print("→ PR-AUC is more informative here due to severe imbalance.")
print("  Gradient Boosting typically achieves the best PR-AUC on this dataset.")

In [ ]:
# ── Confusion matrices for all three models ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, res in zip(axes, all_results):
    cm = confusion_matrix(y_test, res['y_pred'])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm, annot=True, fmt=',.0f', cmap='Blues', ax=ax,
                xticklabels=['Pred Legit', 'Pred Fraud'],
                yticklabels=['Actual Legit', 'Actual Fraud'],
                linewidths=0.5, cbar=False)
    ax.set_title(res['Model'], fontweight='bold')

plt.suptitle('Confusion Matrices (raw counts)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Threshold tuning on the best model ────────────────────────────────────────
# The default 0.5 threshold is almost never optimal for imbalanced problems.
# We sweep thresholds and plot how Precision/Recall trade off.

# Use Gradient Boosting (typically best)
best_model = gb_model
best_probs = gb_results['y_prob']

thresholds = np.arange(0.1, 0.9, 0.01)
f1s, precs, recs = [], [], []

for t in thresholds:
    y_pred_t = (best_probs >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))
    precs.append(precision_score(y_test, y_pred_t, zero_division=0))
    recs.append(recall_score(y_test, y_pred_t, zero_division=0))

optimal_idx = np.argmax(f1s)
optimal_t   = thresholds[optimal_idx]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, f1s,   color='#e74c3c', lw=2, label='F1-score')
ax.plot(thresholds, precs, color='#2980b9', lw=2, label='Precision')
ax.plot(thresholds, recs,  color='#27ae60', lw=2, label='Recall')
ax.axvline(optimal_t, color='black', linestyle='--', lw=1.5,
           label=f'Optimal threshold = {optimal_t:.2f}')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Tuning — Gradient Boosting', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nOptimal threshold : {optimal_t:.2f}")
print(f"F1 at optimal     : {f1s[optimal_idx]:.4f}")
print(f"Precision         : {precs[optimal_idx]:.4f}")
print(f"Recall            : {recs[optimal_idx]:.4f}")

# Re-evaluate best model at optimal threshold
print("\n--- Best Model at Optimal Threshold ---")
gb_results_opt = evaluate_model(gb_model, X_test, y_test,
                                 'Gradient Boosting (Tuned Threshold)',
                                 threshold=optimal_t)

## Section 11 — Model Interpretation & Feature Importance

In [ ]:
# ── Feature importance from Random Forest & Gradient Boosting ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, model, title in [
    (axes[0], rf_model, 'Random Forest'),
    (axes[1], gb_model, 'Gradient Boosting')
]:
    importances = pd.Series(model.feature_importances_, index=FEATURES)
    importances = importances.sort_values(ascending=True)

    colors_bar = [FRAUD_COLOR if v > importances.quantile(0.75) else '#95a5a6'
                  for v in importances.values]
    importances.plot(kind='barh', ax=ax, color=list(reversed(colors_bar)))
    ax.set_title(f'{title} — Feature Importance', fontweight='bold')
    ax.set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

In [ ]:
# ── Top features ranked ───────────────────────────────────────────────────────
gb_importance = pd.Series(gb_model.feature_importances_, index=FEATURES)
print("Top 10 features (Gradient Boosting):")
print(gb_importance.sort_values(ascending=False).head(10).round(4))

In [ ]:
# ── Logistic Regression coefficients ─────────────────────────────────────────
coef_df = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors_coef = [FRAUD_COLOR if c > 0 else LEGIT_COLOR for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_coef, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Logistic Regression — Coefficients (Fraud Direction = Positive)',
             fontweight='bold')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.show()
print("\n→ Red bars push towards fraud prediction; blue bars push away from it.")

## Section 12 — Key Fraud Predictors & Business Interpretation

Based on our models, the most predictive features are consistently:

| Feature | What it captures | Why it signals fraud |
|---|---|---|
| **`error_orig`** | Accounting discrepancy at sender | Fraudsters sometimes manipulate balance records |
| **`orig_zero_after`** | Sender balance goes to exactly 0 | Account is fully drained — classic fraud pattern |
| **`error_dest`** | Accounting discrepancy at recipient | Recipient balance doesn't increase as expected |
| **`log_amount`** | Size of transaction (log scale) | Fraudsters typically move large sums |
| **`dest_zero_before`** | Recipient had zero balance before | Possibly a newly created mule account |
| **`type_TRANSFER`** | Transaction is a TRANSFER | Fraud occurs exclusively in TRANSFER/CASH_OUT |
| **`balance_diff_dest`** | Change in recipient balance | Anomalous changes flag suspicious routing |

### Do these factors make logical sense?

**Yes — they align precisely with known fraud typologies:**

- **Account draining** is the signature of a compromised account. An attacker gains access and immediately moves all funds (hence `orig_zero_after = 1`).

- **Mule accounts** (fresh accounts used once to receive stolen funds) show zero initial balance (`dest_zero_before = 1`). Criminals create these accounts solely as staging points.

- **Balance discrepancies** (`error_orig`, `error_dest`) could indicate test exploits where amounts and balance adjustments were inconsistently applied — a signature of automated fraud bots.

- **Large amounts** make sense economically: the overhead of running a fraud operation is roughly fixed, so fraudsters are incentivised to move as much money as possible per attack.

- The **TRANSFER / CASH_OUT restriction** confirms the synthetic dataset mirrors real-world mobile money fraud: these are the only mechanisms that allow funds to exit the ecosystem.

In [ ]:
# ── Visualise the zero-balance anomaly ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, feat, title in [
    (axes[0], 'orig_zero_after',  'Sender Zero Balance After'),
    (axes[1], 'dest_zero_before', 'Recipient Zero Balance Before'),
]:
    ct = pd.crosstab(df[feat], df['isFraud'], normalize='index') * 100
    ct.columns = ['Legitimate %', 'Fraud %']
    ct.plot(kind='bar', ax=ax, color=[LEGIT_COLOR, FRAUD_COLOR],
            edgecolor='white', width=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Flag (0=No, 1=Yes)')
    ax.set_ylabel('Percentage within group')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(['Legitimate', 'Fraud'])

plt.suptitle('Zero-Balance Flags vs Fraud Class', fontsize=13,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Section 13 — Business Recommendations

Based on everything the model has revealed, here are concrete actions the company should consider:

### Immediate Actions

1. **Focus fraud controls on TRANSFER and CASH_OUT** — All fraud in this dataset happens in these two transaction types. Applying enhanced screening to only these reduces both false positives and computational cost.

2. **Flag full-balance drain events** — Any transaction that would reduce a sender's balance to exactly zero should trigger a hold and secondary verification. This is a very high-precision rule.

3. **Suspicious recipient profiling** — Transactions to recipients with zero starting balance (potential mule accounts) should require additional verification (e.g., OTP, biometric confirmation).

4. **Amount-based velocity checks** — Large TRANSFER/CASH_OUT transactions (above the 90th percentile, ~\$350,000 in this dataset) should have an additional review layer.

### Proactive Monitoring

5. **Account balance reconciliation** — Continuously reconcile accounting: if `newBalance ≠ oldBalance - amount`, flag the discrepancy immediately. This catches manipulation in real-time.

6. **Temporal anomaly detection** — The temporal analysis showed fraud clusters at specific simulation hours. Monitor for unusual spikes in TRANSFER/CASH_OUT volume as an early warning signal.

7. **Customer behaviour profiles** — Build rolling baseline profiles per customer. Flag deviations: first international transfer, transaction at 3am, transfer to unknown new recipient, etc.

### Customer Experience Balance

8. **Tiered response** — Rather than blocking all suspicious transactions, implement a tiered response: low-confidence fraud → soft challenge (OTP); medium → step-up authentication; high → auto-block and alert.

## Section 14 — Infrastructure & System Improvements

### Real-Time Scoring Pipeline

```
Transaction Event
       ↓
  Feature Store  ←── Historical customer data
       ↓
  Feature Engineering (real-time)
       ↓
  ML Model Scoring Engine  (< 50ms SLA)
       ↓
  Risk Score (0–1)
       ↓
  Rule Engine + Threshold Decision
       ↓
  Approve / Challenge / Block
       ↓
  Feedback Loop → Model Retraining
```

### Key Infrastructure Components

| Component | Purpose | Recommended Tools |
|---|---|---|
| **Streaming ingestion** | Capture every transaction event in real-time | Apache Kafka, AWS Kinesis |
| **Feature store** | Maintain up-to-date customer features | Feast, Tecton, Redis |
| **Model serving** | Low-latency inference (<50ms) | FastAPI, TorchServe, SageMaker |
| **Decision engine** | Combine ML score with business rules | Drools, custom rules engine |
| **Case management** | Analyst review queue for flagged transactions | Actimize, custom dashboard |
| **Model monitoring** | Detect drift and degradation | Evidently AI, Seldon |
| **Feedback loop** | Label outcomes to retrain model | MLflow, Airflow |

### Prevention During Infrastructure Updates

When deploying new systems, specific risks arise:
- **Canary deployment:** Roll out the new model to 5% of traffic first; compare fraud catch rates vs baseline
- **Shadow mode testing:** Run the new model in parallel with production for 2–4 weeks without acting on its outputs — validate it would have caught real fraud
- **Rollback plan:** Maintain the previous model version hot-standby; automate rollback if key metrics degrade
- **Data integrity checks:** Any infrastructure change must include validation that transaction data flows are not corrupted

## Section 15 — Measuring Success After Implementation

In [ ]:
# ── Illustrative KPI dashboard framework ─────────────────────────────────────
# In a real deployment, these metrics would be computed on live labelled data.
# Below is a framework showing HOW to structure the monitoring.

# Simulated 30-day metrics (for illustration)
np.random.seed(42)
days = pd.date_range('2024-01-01', periods=30)
fraud_catch_rate  = 0.82 + np.random.randn(30) * 0.02
false_positive_rt = 0.008 + np.random.randn(30) * 0.001
fraud_loss        = 150000 * (1 - fraud_catch_rate) + np.random.randn(30) * 2000

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(days, fraud_catch_rate * 100, color=LEGIT_COLOR, lw=2, marker='o', ms=4)
axes[0].axhline(80, color='red', linestyle='--', lw=1, label='Target ≥ 80%')
axes[0].set_ylabel('Fraud Catch Rate (%)')
axes[0].set_title('Daily Fraud Catch Rate', fontweight='bold')
axes[0].legend()

axes[1].plot(days, false_positive_rt * 100, color=FRAUD_COLOR, lw=2, marker='o', ms=4)
axes[1].axhline(1, color='red', linestyle='--', lw=1, label='Target ≤ 1%')
axes[1].set_ylabel('False Positive Rate (%)')
axes[1].set_title('Daily False Positive Rate', fontweight='bold')
axes[1].legend()

axes[2].bar(days, fraud_loss, color=ACCENT_COLOR, width=0.7)
axes[2].set_ylabel('Estimated Fraud Loss ($)')
axes[2].set_xlabel('Date')
axes[2].set_title('Daily Estimated Fraud Loss (Escaped Fraud)', fontweight='bold')

plt.suptitle('Post-Deployment Monitoring Dashboard (Illustrative)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print("→ In production, these charts are populated with real outcomes from the fraud ops team.")

### KPIs to Track Post-Deployment

| KPI | Target | Measurement Method |
|---|---|---|
| **Fraud Catch Rate (Recall)** | ≥ 80% | % of confirmed fraud cases that were flagged |
| **False Positive Rate** | ≤ 1% of legit transactions | % of good transactions blocked by model |
| **Mean Time to Detection** | ≤ 2 hours | Time from fraud occurrence to flag |
| **Financial Loss Prevented** | Trend downward | Total $ value of caught fraud vs baseline period |
| **Model Drift Score** | PSI < 0.2 | Population Stability Index on feature distributions |
| **Customer Friction Score** | Track NPS | Survey customers who experienced a false positive |

**Model retraining trigger:** If Recall drops >5 percentage points from baseline OR PSI > 0.25 on key features → trigger automated retraining on recent data.

## Section 16 — Limitations & Future Improvements

### Current Limitations

1. **Synthetic data** — PaySim is a simulation. Real transaction data has messier patterns, more complex agent behaviour, and regulatory constraints on data usage.

2. **No graph features** — Fraudsters often operate in rings. Network graph analysis (e.g., who sends money to whom) can reveal fraud rings invisible to row-level models.

3. **No customer history features** — We used only single-transaction features. A 30-day velocity feature (total TRANSFER volume per customer) would likely be highly predictive.

4. **Static model** — Fraud strategies evolve. A model trained today will degrade over time as fraudsters adapt to detection patterns.

5. **Class label quality** — In real data, fraud labels are noisy (some fraud is never reported, some disputes are resolved in the customer's favour).

### Future Improvements

| Improvement | Expected Benefit |
|---|---|
| **Graph Neural Networks** | Detect coordinated fraud rings |
| **Customer velocity features** | Capture unusual behaviour over time windows |
| **Anomaly detection pre-filter** | Isolation Forest / Autoencoder to surface unknowns |
| **Online learning** | Continuously update model with new labelled cases |
| **Ensemble calibration** | Platt scaling / isotonic regression for better probability estimates |
| **Explainable AI (SHAP)** | Provide per-transaction explanations to fraud analysts |
| **Cost-sensitive learning** | Directly optimise for dollar-value of fraud caught, not F1 |

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 60)
print(compare_df.to_string(index=False))
print("\n" + "=" * 60)
print("  KEY FINDINGS")
print("=" * 60)
print("""
1. Fraud occurs ONLY in TRANSFER and CASH_OUT transactions.
2. Account draining (orig_zero_after) is the strongest fraud signal.
3. Balance accounting errors strongly distinguish fraud from legitimate.
4. Gradient Boosting achieves the best Recall-Precision balance.
5. Threshold tuning is essential — default 0.5 is sub-optimal for imbalanced fraud.
6. Real-time balance reconciliation can serve as a rule-based safety net.
""")
print("Notebook complete ✅")